# ohio trade fill investigation

load the real `JDZM_US` fill and compare reference and zerohash best levels around it. ohio is read through the analytics api as recorder `OH01`; this notebook does not write s3 data.

In [ ]:
import os
import sys
from pathlib import Path

_oc = Path(os.getenv('JST_OC_ROOT', Path.home() / 'Desktop/jst/oc')).expanduser()
os.environ.setdefault('JST_ROOT', str(_oc.parent))

_umm_src = str(_oc.parent / 'umm/src')
if _umm_src not in sys.path:
    sys.path.insert(0, _umm_src)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from umm.analytics.market_data import MarketData
from umm.analytics.trading_data import BotData
from umm.analytics.compute.market_snapshot import build_market_snapshot

In [ ]:
symbol = 'PAIR-PUMP-USD'
book = 'JDZM_US'
fill_order_id = 'e7b5edd2-3796-428d-bb69-861fe96df8ec'
start = '2026-10-01 06:35:20.000000'
end = '2026-10-01 06:35:28.000000'
order_start = '2026-10-01 06:30:00.000000'
order_end = '2026-10-01 06:45:00.000000'
feeds = [('OH01', 'COINBASE_FIX'), ('OH01', 'COINBASE_WSS')]

def load_feed(recorder, exchange, timestamp='exchange'):
    return MarketData.load_best_levels(
        symbol=symbol, exchange=exchange, start=start, end=end,
        recorder=recorder, source='influx', timestamp=timestamp,
        stats=True, silent=True,
    )

exchange_views, receive_views = {}, {}
for recorder, exchange in feeds:
    key = f'{recorder}:{exchange}'
    exchange_views[key] = load_feed(recorder, exchange, 'exchange')
    receive_views[key] = load_feed(recorder, exchange, 'receive')
    print(key, len(exchange_views[key]), 'exchange rows;', len(receive_views[key]), 'receive rows')

In [ ]:
snapshot = build_market_snapshot(
    symbol=symbol, exchange='COINBASE_FIX',
    start=start, end=end, book=book, jst_sym=symbol,
    source='influx', max_workers=1, perf=True,
)
fills = snapshot.fills.copy()
fills['timestamp'] = pd.to_datetime(fills['timestamp'], utc=True)
fill_mask = (
    fills['exchange_oid'].astype(str).eq(fill_order_id)
    | fills['client_id'].astype(str).eq(fill_order_id)
)
fill = fills[fill_mask].copy()
print('snapshot timings:', snapshot.timings)
print('fills in window:', len(fills), 'matching order:', len(fill))
display(fill if not fill.empty else fills)

In [ ]:
order_lifetime = BotData.load_order_lifetime(
    book, order_start, order_end, symbol=symbol, source='influx',
    max_workers=1,
)
order_events = order_lifetime[
    order_lifetime['client_id'].astype(str).eq(fill_order_id)
].sort_values('timestamp')
print('matching order events:', len(order_events))
display(order_events)

In [ ]:
reference_exchange_view = exchange_views['OH01:COINBASE_FIX']
reference_receive_view = receive_views['OH01:COINBASE_FIX']
book_exchange_view = MarketData.load_best_levels(
    symbol=symbol, exchange='ZEROHASH', start=start, end=end,
    recorder='TY03', source='influx', timestamp='exchange',
    stats=True, silent=True,
)
book_receive_view = MarketData.load_best_levels(
    symbol=symbol, exchange='ZEROHASH', start=start, end=end,
    recorder='TY03', source='influx', timestamp='receive',
    stats=True, silent=True,
)
print('exchange rows:', len(reference_exchange_view), 'reference;', len(book_exchange_view), 'book')
print('receive rows:', len(reference_receive_view), 'reference;', len(book_receive_view), 'book')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
plot_views = [
    (axes[0], reference_exchange_view, book_exchange_view, 'exchange timestamps'),
    (axes[1], reference_receive_view, book_receive_view, 'receive timestamps'),
]
for ax, reference_view, book_view, timestamp_label in plot_views:
    if not reference_view.empty:
        ax.step(reference_view.index, reference_view['bid_price'], where='post', color='green', label='ref bid')
        ax.step(reference_view.index, reference_view['ask_price'], where='post', color='red', label='ref ask')
    if not book_view.empty:
        ax.step(book_view.index, book_view['bid_price'], where='post', color='lightgreen', linestyle='--', label='book bid')
        ax.step(book_view.index, book_view['ask_price'], where='post', color='lightcoral', linestyle='--', label='book ask')
    if not fill.empty:
        fill_time = pd.to_datetime(fill['timestamp'], utc=True).iloc[0].tz_localize(None)
        fill_price = float(fill['price'].iloc[0])
        ax.plot(fill_time, fill_price, marker='v', color='red', markersize=10, label='sell maker fill')
        ax.axvline(fill_time, color='black', alpha=0.25, linewidth=1)
    if not order_events.empty:
        alive_time = pd.to_datetime(order_events['timestamp'], utc=True).iloc[0]
        ax.axvline(alive_time, color='purple', alpha=0.35, linestyle=':', label='order alive')
    center = fill_time if not fill.empty else pd.Timestamp('2026-10-01 06:35:24.189')
    ax.set_xlim(center - pd.Timedelta(milliseconds=60), center + pd.Timedelta(milliseconds=60))
    ax.set_title(f'jdzm / pair-pump-usd — {timestamp_label}')
    ax.set_xlabel('time')
    ax.grid(alpha=0.25)
    ax.legend(fontsize='small')
axes[0].set_ylabel('price')
plt.tight_layout()

In [ ]:
quote_checks = []
if not order_events.empty:
    first_order_ts = pd.to_datetime(order_events['timestamp'], utc=True).min().tz_localize(None)
    fill_price_value = float(fill['price'].iloc[0]) if not fill.empty else float('nan')
    for key, frame in exchange_views.items():
        frame_before = frame[frame.index <= first_order_ts]
        if frame_before.empty:
            continue
        quote = frame_before.iloc[-1]
        quote_ts = pd.Timestamp(frame_before.index[-1])
        mid = (float(quote['bid_price']) + float(quote['ask_price'])) / 2
        quote_checks.append({
            'feed': key,
            'first_order_event': first_order_ts,
            'quote_timestamp': quote_ts,
            'order_minus_quote_ms': (first_order_ts - quote_ts).total_seconds() * 1000,
            'bid_price': quote['bid_price'],
            'ask_price': quote['ask_price'],
            'mid_price': mid,
            'fill_minus_mid_bps': (fill_price_value - mid) / mid * 1e4,
        })
quote_checks = pd.DataFrame(quote_checks)
display(quote_checks)

In [ ]:
latency_rows = []
for key, frame in receive_views.items():
    if frame.empty or 'exchange_timestamp' not in frame.columns:
        continue
    exchange_ts = pd.to_datetime(frame['exchange_timestamp'], utc=True)
    receive_ts = pd.to_datetime(frame.index, utc=True)
    latency_rows.append(pd.DataFrame({
        'feed': key,
        'exchange_timestamp': exchange_ts,
        'receive_timestamp': receive_ts,
        'recv_minus_exchange_ms': (receive_ts - exchange_ts).dt.total_seconds() * 1000,
        'bid_price': frame['bid_price'].to_numpy(),
        'ask_price': frame['ask_price'].to_numpy(),
    }))
latency = pd.concat(latency_rows, ignore_index=True) if latency_rows else pd.DataFrame()
latency_summary = (latency.groupby('feed')['recv_minus_exchange_ms']
                   .agg(['count', 'median', 'mean', 'max'])
                   .sort_index()) if not latency.empty else pd.DataFrame()
display(latency_summary)
display(latency.sort_values(['feed', 'exchange_timestamp']).head())